# Trajectory Optimization: Shortest Reliable Path

**Primary principle:**
THE SHORTEST RUN IS NOT NECESSARILY THE BEST RUN.
OPTIMIZE COST / LATENCY / WORK
SUBJECT TO
QUALITY + SAFETY + GROUNDING CONSTRAINTS.

This notebook demonstrates how to analyze agent trajectories for unnecessary work, remove duplicate reads, apply safe caching, parallelize independent tools, and bound retries and reflections, all without violating the evaluation boundaries established in Course 05.

## Part 1: Baseline Typed Trajectory

We start with a baseline trajectory from the Northstar EU checkout latency scenario. The baseline has unnecessary work: it reads health twice, logs twice, and runs everything sequentially.

In [ ]:
import sys
import os
import time
sys.path.append(os.path.abspath("curriculum/intermediate/06-trajectory-optimization"))

from policy import (
    StepType, ResultStatus, ToolEffect, StepClassification,
    OptimizationType, ToolDefinition, TrajectoryStep, Trajectory,
    TrajectoryEvalContext, TrajectoryMetrics, OptimizationCandidate, 
    OptimizationPlan, OptimizationResult, TrajectoryComparison,
    classify_steps, can_parallelize, is_valid_cache_hit, compute_metrics, 
    optimization_regression_gate, should_stop, compare_trajectories,
    find_optimization_candidates, apply_optimization, evaluate_optimization
)

tools = {
    "get_service_health": ToolDefinition(name="get_service_health", effect=ToolEffect.READ, supports_parallel=True, cacheable=True, rate_limit_group="db"),
    "query_logs": ToolDefinition(name="query_logs", effect=ToolEffect.READ, supports_parallel=True, cacheable=True, rate_limit_group="logs"),
    "get_deployment": ToolDefinition(name="get_deployment", effect=ToolEffect.READ, supports_parallel=True, cacheable=True, rate_limit_group="k8s"),
    "restart_service": ToolDefinition(name="restart_service", effect=ToolEffect.WRITE, supports_parallel=False, cacheable=False),
    "get_customer": ToolDefinition(name="get_customer", effect=ToolEffect.READ, supports_parallel=True, cacheable=True),
    "get_orders": ToolDefinition(name="get_orders", effect=ToolEffect.READ, supports_parallel=True, cacheable=True)
}

context = TrajectoryEvalContext(
    expected_outcome="Identify database connection pool exhaustion in EU-West.",
    available_evidence_ids=["health_data_eu", "log_data_eu", "deploy_eu", "customer_1", "order_1"],
    required_evidence_ids=["health_data_eu", "log_data_eu"],
    forbidden_tools=["restart_service"],
    tenant_id="northstar",
    tools=tools,
    dependency_graph={
        "get_customer_1": ["get_orders_1"]  # Orders depend on Customer
    }
)

baseline = Trajectory(
    run_id="run-baseline-01",
    tenant_id="northstar",
    final_answer="Identify database connection pool exhaustion in EU-West.",
    agent_version="1.0", prompt_version="1.0", model_version="1.0", tool_version="1.0", policy_version="1.0", dataset_version="1.0",
    steps=[
        TrajectoryStep(step_id="step_1", step_type=StepType.TOOL_CALL, tool_name="get_service_health", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=500, cost_usd=0.01),
        TrajectoryStep(step_id="step_2", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, evidence_ids=["health_data_eu"], latency_ms=10, cost_usd=0),
        TrajectoryStep(step_id="step_3", step_type=StepType.TOOL_CALL, tool_name="query_logs", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=1200, cost_usd=0.02),
        TrajectoryStep(step_id="step_4", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, evidence_ids=["log_data_eu"], latency_ms=10, cost_usd=0),
        TrajectoryStep(step_id="step_5", step_type=StepType.TOOL_CALL, tool_name="get_service_health", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=500, cost_usd=0.01),
        TrajectoryStep(step_id="step_6", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, evidence_ids=["health_data_eu"], latency_ms=10, cost_usd=0),
        TrajectoryStep(step_id="step_7", step_type=StepType.TOOL_CALL, tool_name="query_logs", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=1200, cost_usd=0.02),
        TrajectoryStep(step_id="step_8", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, evidence_ids=["log_data_eu"], latency_ms=10, cost_usd=0),
        TrajectoryStep(step_id="step_9", step_type=StepType.TOOL_CALL, tool_name="get_deployment", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=600, cost_usd=0.01),
        TrajectoryStep(step_id="step_10", step_type=StepType.TOOL_RESULT, result_status=ResultStatus.SUCCESS, evidence_ids=["deploy_eu"], latency_ms=10, cost_usd=0),
    ]
)
print("Baseline trajectory loaded.")

## Part 2 & 3: Instrumentation, Metrics, and Step Classification

Before optimizing, we must classify each step. Is this a `DUPLICATE_READ`, a `SIDE_EFFECT`, or `REQUIRED_EVIDENCE`?

In [ ]:
classified_baseline = classify_steps(baseline, context.tools)

print("Step Classifications:")
for s in classified_baseline.steps:
    if s.step_type == StepType.TOOL_CALL:
        print(f"  {s.tool_name} -> {s.classification.value}")
        
baseline_metrics = compute_metrics(baseline, context)
print(f"\nBaseline Critical Path: {baseline_metrics.critical_path_ms}ms, Total Work: {baseline_metrics.total_work_ms}ms")


## Part 4: Duplicate Removal

We see duplicate reads. Can we just remove them? Yes, because they are `DUPLICATE_READ`. If they were a repeated WRITE, we would NOT remove them as a simple optimization—that would mask an idempotency bug!

In [ ]:
candidates = find_optimization_candidates(baseline, context)

# Apply removal
dup_candidate = next(c for c in candidates if c.optimization_type == OptimizationType.REMOVE_DUPLICATE_READ)
optimized_no_dups = apply_optimization(baseline, dup_candidate)

result_no_dups = evaluate_optimization(baseline, optimized_no_dups, dup_candidate, context)
print(f"Duplicate removal accepted? {result_no_dups.accepted}. Saved: {result_no_dups.actual_latency_savings_ms}ms")


## Part 5: Dependency-Aware Parallelism

Can we parallelize the independent reads? Total work vs wall-clock latency.

In [ ]:
# 1. Independent reads can be parallelized
s_health = [s for s in optimized_no_dups.steps if s.tool_name == "get_service_health"][0]
s_logs = [s for s in optimized_no_dups.steps if s.tool_name == "query_logs"][0]
print(f"Can parallelize health and logs? {can_parallelize(s_health, s_logs, context)}")

# 2. Dependent reads CANNOT be parallelized
s_cust = TrajectoryStep(step_id="get_customer_1", step_type=StepType.TOOL_CALL, tool_name="get_customer", target_tenant_id="northstar", latency_ms=100, cost_usd=0)
s_ord = TrajectoryStep(step_id="get_orders_1", step_type=StepType.TOOL_CALL, tool_name="get_orders", target_tenant_id="northstar", latency_ms=100, cost_usd=0)
print(f"Can parallelize customer and orders? {can_parallelize(s_cust, s_ord, context)}")


## Part 6: Caching

Cache safe deterministic reads only. Verify explicit arguments, source version, and TTL bounds.

In [ ]:
now = time.time()
cache_entry = {"policy_version": "1.0", "expires_at": now + 1000, "source_version": "v1.2", "arguments": "{'service': 'checkout'}"}
s_test = TrajectoryStep(step_id="c1", step_type=StepType.TOOL_CALL, tool_name="get_service_health", arguments={"service":"checkout"}, target_tenant_id="northstar", latency_ms=10, cost_usd=0, source_version="v1.2")

print("Valid cache hit?", is_valid_cache_hit(s_test, cache_entry, context, now))
print("Valid cache hit (stale TTL)?", is_valid_cache_hit(s_test, {"policy_version": "1.0", "expires_at": now - 100, "source_version": "v1.2", "arguments": "{'service': 'checkout'}"}, context, now))
print("Valid cache hit (cross-tenant)?", is_valid_cache_hit(TrajectoryStep(**{**s_test.model_dump(), "target_tenant_id": "globex"}), cache_entry, context, now))
print("Valid cache hit (different source)?", is_valid_cache_hit(TrajectoryStep(**{**s_test.model_dump(), "source_version": "v1.3"}), cache_entry, context, now))


## Part 7 & 8 & 9: Reflection Limits, Early Stopping, Retry Semantics

- **Reflection:** Bound open-ended loops. Use threshold-based early stopping.
- **Early Stopping:** If evidence `{'health', 'logs'}` is required, stop retrieving once it's met. Don't fetch `deployment` just because budget remains.
- **Retries:** Only retry TIMEOUT or REPAIRABLE schema errors. Do NOT retry POLICY_BLOCKED or AUTH_BLOCKED.

In [ ]:
# Early Stopping
print("Should stop (health)?", should_stop({"health_data_eu"}, set(context.required_evidence_ids)))
print("Should stop (health + logs)?", should_stop({"health_data_eu", "log_data_eu"}, set(context.required_evidence_ids)))


## Part 10 & 11: Baseline vs Optimized Comparison & Regression Rejection

A faster trajectory that removes REQUIRED evidence MUST FAIL the regression gate.

In [ ]:
# Candidate that removes `get_deployment` (which isn't required evidence). This is a GOOD optimization.
optimized_early_stop_steps = [s for s in optimized_no_dups.steps if "get_deployment" not in (s.tool_name or "")]
good_candidate = Trajectory(**{**optimized_no_dups.model_dump(), "steps": optimized_early_stop_steps})

res_good = evaluate_optimization(baseline, good_candidate, OptimizationCandidate(
    optimization_type=OptimizationType.EARLY_STOP, affected_steps=["step_9"], rationale="Not required", expected_latency_savings_ms=600, expected_cost_savings_usd=0.01, constraints_checked=[]
), context)
print(f"Early stop accepted? {res_good.accepted} (Saved: {res_good.actual_latency_savings_ms}ms)")

# Candidate that removes `query_logs` (which IS required evidence). This is a BAD optimization.
optimized_bad_steps = [s for s in optimized_early_stop_steps if "query_logs" not in (s.tool_name or "")]
bad_candidate = Trajectory(**{**optimized_no_dups.model_dump(), "steps": optimized_bad_steps})

res_bad = evaluate_optimization(baseline, bad_candidate, OptimizationCandidate(
    optimization_type=OptimizationType.EARLY_STOP, affected_steps=["step_3"], rationale="Save latency", expected_latency_savings_ms=1200, expected_cost_savings_usd=0.02, constraints_checked=[]
), context)
print(f"Bad optimization accepted? {res_bad.accepted} (Reason: {res_bad.rejection_reason})")


## Part 12: Pareto Trade-offs

Lowest latency, lowest cost, and highest reliability might be three completely different execution paths. Accept that optimization is a Pareto frontier, not a single global maximum.

## Part 13: Optional Deep Dive: DSPy Program Optimization

We can use programmatic compilers like DSPy to optimize the prompts themselves to encourage the model toward the optimized trajectory shape, but DSPy alone is NOT trajectory optimization—it is LM program optimization against a metric.

In [ ]:
# 1. Define DSPy Signature
# class OptimizerSignature(dspy.Signature):
#     ...
#
# 2. Define Teleprompter
# from dspy.teleprompt import BootstrapFewShot
# ...
